In [1]:
!pip install kagglehub\[pandas-datasets\]
!pip install pydot
!pip install graphviz

In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import kagglehub
from kagglehub import KaggleDatasetAdapter

# OBTENER INFORMACION

In [3]:
"""
Carga y preprocesamiento del conjunto de datos de reseñas de Yelp.
El objetivo es preparar tres particiones (train, validation, test)
como tf.data.Dataset listos para entrenar un modelo de clasificación.
"""

RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

# ————— Carga de datos —————
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "omkarsabnis/yelp-reviews-dataset",
    "yelp.csv",
)

# Conservamos solo las columnas relevantes
df = df[['text', 'stars']]

# ————— Tokenización simple y cálculo de longitud —————
# Creamos una nueva columna 'token_count' con el conteo de palabras por reseña
df['token_count'] = df['text'].astype(str).apply(lambda txt: len(txt.split()))

# Determinamos el percentil 95 de longitud para posibles truncamientos posteriores
percentile_95 = np.percentile(df['token_count'], 95)
max_length = int(round(percentile_95))
max = np.max(df['token_count'])

# ————— Preparación de features y target —————
# Restamos 1 a 'stars' para convertirlas en etiquetas 0–4
target = df.pop('stars') - 1
features = df['text']

# ————— División en train / validation / test —————
x_temp, x_test, y_temp, y_test = train_test_split(
    features, target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

# ————— Conversión a tf.data.Dataset —————
raw_train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
raw_val_ds   = tf.data.Dataset.from_tensor_slices((x_val,   y_val))
raw_test_ds  = tf.data.Dataset.from_tensor_slices((x_test,  y_test))

<ipython-input-3-8757721f9337>:12: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 3.49M/3.49M [00:00<00:00, 38.3MB/s]

Extracting zip of yelp.csv...


In [4]:
print(f"Tamaño del conjunto de entrenamiento: {len(raw_train_ds)}")
print(f"Tamaño del conjunto de validación: {len(raw_val_ds)}")
print(f"Tamaño del conjunto de prueba: {len(raw_test_ds)}")

Tamaño del conjunto de entrenamiento: 6400
Tamaño del conjunto de validación: 1600
Tamaño del conjunto de prueba: 2000


In [ ]:
import collections

# Contamos cuántos ejemplos hay por clase en raw_train_ds
label_counter = collections.Counter()

for _, label in raw_train_ds:
    label_counter[int(label.numpy())] += 1

# Mostramos los resultados
for label, count in sorted(label_counter.items()):
    print(f"Clase {label}: {count} ejemplos")

Clase 0: 472 ejemplos
Clase 1: 625 ejemplos
Clase 2: 920 ejemplos
Clase 3: 2240 ejemplos
Clase 4: 2143 ejemplos


# MODELO

In [ ]:
# ————— Configuración de la capa de vectorización de texto —————

# Número máximo de palabras únicas que el modelo considerará (las más frecuentes)
VOCAB_SIZE = 20000

# Longitud fija para todas las secuencias de entrada (se recortarán o rellenarán hasta este tamaño)
sequence_length = max_length

# Creamos una capa TextVectorization que transforma texto crudo en secuencias de enteros
vectorize_layer = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',   # convierte todo a minúsculas y elimina puntuación
    max_tokens=VOCAB_SIZE,                       # restringe el vocabulario a las 20 000 tokens más frecuentes
    output_mode='int',                           # cada token se mapea a un entero único
    output_sequence_length=sequence_length       # fuerza todas las secuencias a tener la misma longitud
)

# Adaptamos la capa al corpus completo de textos para construir el índice de tokens
# Esto analiza la frecuencia de cada palabra y crea el vocabulario interno
vectorize_layer.adapt(features)

In [ ]:
def vectorized_text(text: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    """
    Convierte un tensor de texto crudo a secuencia de enteros usando la capa TextVectorization.

    Args:
        text: Tensor de tipo string con la reseña original.
        label: Tensor con la etiqueta (estrellas menos 1).

    Returns:
        Tupla (texto_vectorizado, label) donde texto_vectorizado es un tensor de enteros
        de longitud fija y label permanece inalterado.
    """
    return vectorize_layer(text), label

# Aplicamos la función de vectorización a cada partición para obtener datasets listos para entrenamiento
train_ds = raw_train_ds.map(vectorized_text)
val_ds   = raw_val_ds.map(vectorized_text)
test_ds  = raw_test_ds.map(vectorized_text)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Generamos la lista de etiquetas del dataset
y_train_list = list(y_train)

# Calculamos pesos automáticamente
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_list),
    y=y_train_list
)

# Lo convertimos en diccionario para pasarlo al modelo
class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(2.711864406779661), 1: np.float64(2.048), 2: np.float64(1.391304347826087), 3: np.float64(0.5714285714285714), 4: np.float64(0.5972935137657489)}


# RNN

In [ ]:
initializer = tf.keras.initializers.GlorotUniform()

# ————— Constantes de configuración —————
VOCAB_SIZE            = 20000
EMBEDDING_DIM         = 300
MAX_SEQUENCE_LENGTH   = max_length
NUM_CLASSES           = 5
LEARNING_RATE         = 1e-3

# ————— Entrada —————
inputs = tf.keras.Input(
    shape=(MAX_SEQUENCE_LENGTH,),
    dtype=tf.int64,
    name="input_text_tokens"
)

# ————— Embedding —————
x = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH,
    name="embedding"
)(inputs)

# ————— Primera capa SimpleRNN —————
x = tf.keras.layers.SimpleRNN(
    units=128,
    return_sequences=True,  # 👈 Devuelve toda la secuencia para apilar
    recurrent_dropout=0.2,
    kernel_initializer=initializer,
    recurrent_initializer=initializer,
    name="simple_rnn_1"
)(x)

# ————— Segunda capa SimpleRNN —————
x = tf.keras.layers.SimpleRNN(
    units=64,
    return_sequences=False,  # 👈 Esta ya colapsa a una salida
    recurrent_dropout=0.2,
    kernel_initializer=initializer,
    recurrent_initializer=initializer,
    name="simple_rnn_2"
)(x)

# ————— Normalización y densa —————
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)

# ————— Capa de salida —————
outputs = tf.keras.layers.Dense(
    units=NUM_CLASSES,
    activation="softmax",
    name="output_probabilities"
)(x)

# ————— Construcción del modelo —————
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="stacked_simple_rnn")

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()


Model: "stacked_simple_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_text_tokens (InputLayer)  │ (None, 348)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 348, 300)       │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 348, 128)       │        54,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probabilities (Dense)    │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,072,005 (23.16 MB)

 Trainable params: 6,071,877 (23.16 MB)

 Non-trainable params: 128 (512.00 B)

In [ ]:
# ————— Configuración de hiperparámetros de entrenamiento —————

BATCH_SIZE = 32
EPOCHS     = 10
PATIENCE   = 3

# ————— Callback: EarlyStopping —————
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Monitorea la pérdida en validación
    patience=PATIENCE,       # Detiene si no hay mejora tras N épocas
    restore_best_weights=True  # Restaura los pesos del mejor modelo
)

# ————— Entrenamiento del modelo —————
history_rnn = model.fit(
    train_ds.batch(BATCH_SIZE),
    validation_data=val_ds.batch(BATCH_SIZE),
    epochs=EPOCHS,
    # callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 85s 406ms/step - accuracy: 0.2802 - loss: 1.5583 - val_accuracy: 0.3625 - val_loss: 1.4503
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 80s 397ms/step - accuracy: 0.3477 - loss: 1.4616 - val_accuracy: 0.3400 - val_loss: 1.4575
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 78s 378ms/step - accuracy: 0.3581 - loss: 1.4519 - val_accuracy: 0.3325 - val_loss: 1.4568
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 384ms/step - accuracy: 0.3632 - loss: 1.4413 - val_accuracy: 0.3525 - val_loss: 1.4471
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 81s 378ms/step - accuracy: 0.3544 - loss: 1.4423 - val_accuracy: 0.3494 - val_loss: 1.4485
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 383ms/step - accuracy: 0.3530 - loss: 1.4382 - val_accuracy: 0.3181 - val_loss: 1.4494
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 80s 376ms/step - accuracy: 0.3441 - loss: 1.4388 - val_accuracy: 0.3487 - val_loss: 1.4433
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 78s 388ms/step - accuracy: 0.3537 - loss: 1

# LSTM

In [ ]:
# ————— Constantes de configuración —————
VOCAB_SIZE            = 20000
EMBEDDING_DIM         = 64
MAX_SEQUENCE_LENGTH   = max_length
NUM_CLASSES           = 5
LEARNING_RATE         = 1e-3

# ————— Entrada —————
inputs = tf.keras.Input(
    shape=(MAX_SEQUENCE_LENGTH,),
    dtype=tf.int64,
    name="input_text_tokens"
)

# ————— Embedding —————
x = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH,
    name="embedding"
)(inputs)

# ————— LSTM mejorado —————
x = tf.keras.layers.LSTM(
    units=128,
    dropout=0.3,
    recurrent_dropout=0.3,
    return_sequences=False,
    name="lstm"
)(x)

# ————— Capa densa intermedia —————
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)

# ————— Capa de salida —————
outputs = tf.keras.layers.Dense(
    units=NUM_CLASSES,
    activation="softmax",
    name="output_probabilities"
)(x)

# ————— Modelo —————
model_lstm = tf.keras.Model(inputs=inputs, outputs=outputs, name="review_lstm_classifier")

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model_lstm.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model_lstm.summary()

Model: "review_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_text_tokens (InputLayer)  │ (None, 348)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 348, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probabilities (Dense)    │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,397 (5.29 MB)

 Trainable params: 1,387,397 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ————— Configuración de hiperparámetros de entrenamiento —————

BATCH_SIZE = 128
EPOCHS     = 10
PATIENCE   = 3

# ————— Callback: EarlyStopping —————
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Monitorea la pérdida en validación
    patience=PATIENCE,       # Detiene si no hay mejora tras N épocas
    restore_best_weights=True  # Restaura los pesos del mejor modelo
)

# ————— Entrenamiento del modelo —————
history_lstm = model_lstm.fit(
    train_ds.batch(BATCH_SIZE),   # Baraja y agrupa en lotes
    validation_data=val_ds.batch(BATCH_SIZE),                 # Lotes para validación
    epochs=EPOCHS,                                            # Máximo de épocas
    # callbacks=[early_stopping],                               # Early stopping integrado
    verbose=1,
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.2091 - loss: 1.5890 - val_accuracy: 0.0850 - val_loss: 1.6152
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.1866 - loss: 1.5889 - val_accuracy: 0.3325 - val_loss: 1.6075
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.2436 - loss: 1.5879 - val_accuracy: 0.3300 - val_loss: 1.6082
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.2890 - loss: 1.5871 - val_accuracy: 0.3237 - val_loss: 1.6000
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.2556 - loss: 1.5878 - val_accuracy: 0.3331 - val_loss: 1.6109
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 98s 2s/step - accuracy: 0.2208 - loss: 1.5820 - val_accuracy: 0.3225 - val_loss: 1.6199
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 98s 2s/step - accuracy: 0.2408 - loss: 1.5766 - val_accuracy: 0.3250 - val_loss: 1.6316
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.2770 - loss: 1.5726 - val_accuracy: 0.3231 - val

# BiLSTM

In [ ]:
initializer = tf.keras.initializers.GlorotUniform()

# ————— Constantes de configuración —————
VOCAB_SIZE            = 20000
EMBEDDING_DIM         = 300  # Puedes bajarlo a 128 si observas sobreajuste
MAX_SEQUENCE_LENGTH   = max_length
NUM_CLASSES           = 5
LEARNING_RATE         = 1e-3

# ————— Entrada —————
inputs = tf.keras.Input(
    shape=(MAX_SEQUENCE_LENGTH,),
    dtype=tf.int64,
    name="input_text_tokens"
)

# ————— Embedding + SpatialDropout —————
x = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH,
    name="embedding"
)(inputs)

x = tf.keras.layers.SpatialDropout1D(0.3)(x)  # Mejora robusta para NLP

# ————— Bidirectional LSTM 1 —————
x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(
        units=128,
        return_sequences=True,
        dropout=0.3,
        recurrent_dropout=0.3,
        kernel_initializer=initializer,
        recurrent_initializer=initializer,
        name="lstm_1"
    )
)(x)

# ————— Bidirectional LSTM 2 —————
x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(
        units=64,
        return_sequences=False,
        dropout=0.2,
        recurrent_dropout=0.2,
        kernel_initializer=initializer,
        recurrent_initializer=initializer,
        name="lstm_2"
    )
)(x)

# ————— BatchNorm + Dense + Dropout + L2 —————
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(
    64,
    kernel_initializer='he_uniform',
    activation='relu',
    kernel_regularizer=regularizers.l2(1e-4)
)(x)
x = tf.keras.layers.Dropout(0.5)(x)

# ————— Output —————
outputs = tf.keras.layers.Dense(
    units=NUM_CLASSES,
    activation='softmax',
    name="output_probabilities"
)(x)

# ————— Modelo —————
model_bilstm = tf.keras.Model(inputs=inputs, outputs=outputs, name="review_bilstm_classifier")

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model_bilstm.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model_bilstm.summary()

Model: "review_bilstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_text_tokens (InputLayer)  │ (None, 348)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 348, 300)       │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 348, 300)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 348, 256)       │       439,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probabilities (Dense)    │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,612,741 (25.23 MB)

 Trainable params: 6,612,485 (25.22 MB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
# ————— Configuración de hiperparámetros de entrenamiento —————

BATCH_SIZE = 128
EPOCHS     = 10
PATIENCE   = 3

# ————— Callback: EarlyStopping —————
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Monitorea la pérdida en validación
    patience=PATIENCE,       # Detiene si no hay mejora tras N épocas
    restore_best_weights=True  # Restaura los pesos del mejor modelo
)

# ————— Entrenamiento del modelo —————
history_bilstm = model_bilstm.fit(
    train_ds.batch(BATCH_SIZE),
    validation_data=val_ds.batch(BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 523s 10s/step - accuracy: 0.2564 - loss: 1.8528 - val_accuracy: 0.3419 - val_loss: 1.5626
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 515s 10s/step - accuracy: 0.3952 - loss: 1.4582 - val_accuracy: 0.3419 - val_loss: 1.5515
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 517s 10s/step - accuracy: 0.4490 - loss: 1.3038 - val_accuracy: 0.3681 - val_loss: 1.5118
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 561s 10s/step - accuracy: 0.5344 - loss: 1.1253 - val_accuracy: 0.3631 - val_loss: 1.4754
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 511s 10s/step - accuracy: 0.6183 - loss: 0.9671 - val_accuracy: 0.3931 - val_loss: 1.3850
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 558s 10s/step - accuracy: 0.6901 - loss: 0.8024 - val_accuracy: 0.3988 - val_loss: 1.3258
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 562s 10s/step - accuracy: 0.7552 - loss: 0.6667 - val_accuracy: 0.4275 - val_loss: 1.3678
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 566s 10s/step - accuracy: 0.8107 - loss: 0.5311 - val_accuracy: 0.

In [ ]:
# ————— Evaluación del modelo en el conjunto de prueba —————

TEST_BATCH_SIZE = 32

# Evalúa la pérdida y precisión de los modelos usando el dataset de prueba
test_results_rnn = model.evaluate(
    test_ds.batch(TEST_BATCH_SIZE),
    verbose=1
)

test_results_lstm = model_lstm.evaluate(
    test_ds.batch(TEST_BATCH_SIZE),
    verbose=1
)

test_results_bilstm = model_bilstm.evaluate(
    test_ds.batch(TEST_BATCH_SIZE),
    verbose=1
)

print(f"Test loss: {test_results[0]:.4f} — Test accuracy: {test_results[1]:.4f}")
print(f"Test loss: {test_results_lstm[0]:.4f} — Test accuracy: {test_results_lstm[1]:.4f}")
print(f"Test loss: {test_results_bilstm[0]:.4f} — Test accuracy: {test_results_bilstm[1]:.4f}")

In [ ]:
def graficar_entrenamiento(history):
    plt.figure(figsize=(12, 4))

    # Subplot 1: Pérdida (loss)
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Pérdida de entrenamiento')
    plt.plot(history.history['val_loss'], label='Pérdida de validación')
    plt.title('Evolución de la Pérdida')
    plt.xlabel('Época')
    plt.ylabel('Pérdida')
    plt.legend()

    # Subplot 2: Exactitud (accuracy)
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Exactitud de entrenamiento')
    plt.plot(history.history['val_accuracy'], label='Exactitud de validación')
    plt.title('Evolución de la Exactitud')
    plt.xlabel('Época')
    plt.ylabel('Exactitud')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ————— Graficar resultados de entrenamiento y validación —————
graficar_entrenamiento(history_rnn)
graficar_entrenamiento(history_lstm)
graficar_entrenamiento(history_bilstm)